# IDSC2026 Brugada-HUCA Classification

## 0. Intro

**Objective**

Classify Brugada Syndrome versus Normal ECG recordings from the Brugada HUCA dataset.

**Strategy Pipeline**

1. Locate the dataset automatically and validate the dataset structure and expected recording specification.
2. Verify metadata counts, WFDB coverage, 12-lead availability, 100 Hz sampling, and 12-second duration.
3. Explore class balance and document the Brugada-vs-Normal class imbalance.
4. Apply notebook-level preprocessing after loading the raw WFDB signals, then build engineered features and median-beat representations.
5. Create one shared stratified patient split so every model is compared fairly.
6. Train 11 models with imbalance-aware weighting, threshold tuning, and minority-class-sensitive validation.
7. Rank the models with an imbalance-aware priority that emphasizes PR-AUC, F1, sensitivity, balanced accuracy, and Brugada detection.
8. Save all plots, predictions, and the final summary to the working output folders.

**Why this approach**

- All models below use the same patient split data so the comparison stays fair.
- The notebook checks that the dataset matches the expected structure: 363 subjects, 76 Brugada cases, 287 Normal controls, 12 leads, 100 Hz, and 12-second records.
- The Brugada class is the minority class, so the pipeline uses imbalance-aware training and ranking instead of relying on accuracy alone.
- The dataset is loaded as raw ECG data, so the filtering and normalization in this notebook are explicit post-load preprocessing choices.
- Feature-based models keep the ECG morphology interpretation strong.
- End-to-end models learn directly from the median beat signal.
- Transfer learning stays fully in-domain with Brugada HUCA only, so it does not rely on external labeled datasets.


## 1. Initialization

This section prepares the dependencies, the base configuration, and the automatic dataset discovery logic.


### Step 1 - Install Required Packages

Install the required packages only when they are missing from the current environment.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "wfdb": "wfdb",
    "xgboost": "xgboost",
    "tensorflow": "tensorflow",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


### Step 2 - Import Libraries and Prepare Output Folders

Import the core libraries, set the random seed, and prepare the `plots/` and `predict/` folders inside `/kaggle/working/` when the notebook runs on Kaggle.


In [ ]:
import gc
import os
import random
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import wfdb

from IPython.display import display
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
from scipy.stats import kurtosis, skew
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks, layers, models, optimizers
from tqdm.auto import tqdm
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

randomState = 42
fsExpected = 100
standardLeads = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)

projectRoot = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
plotDir = projectRoot / "plots"
predictionDir = projectRoot / "predict"
plotDir.mkdir(exist_ok=True)
predictionDir.mkdir(exist_ok=True)

_saved_plot_counts = {}

def sanitize_name(text):
    text = re.sub(r"[^A-Za-z0-9]+", "_", str(text).strip()).strip("_").lower()
    return text or "artifact"

def move_legends_outside(fig=None):
    fig = fig or plt.gcf()
    for ax in fig.axes:
        legend = ax.get_legend()
        if legend is None:
            continue
        handles, labels = ax.get_legend_handles_labels()
        if not handles:
            continue
        legend_title = legend.get_title().get_text() if legend.get_title() is not None else None
        legend.remove()
        ax.legend(
            handles,
            labels,
            loc="upper left",
            bbox_to_anchor=(1.02, 1),
            borderaxespad=0.0,
            title=legend_title if legend_title else None,
        )

def save_plot(plot_name):
    fig = plt.gcf()
    move_legends_outside(fig)
    safe_name = sanitize_name(plot_name)
    current_count = _saved_plot_counts.get(safe_name, 0) + 1
    _saved_plot_counts[safe_name] = current_count
    suffix = "" if current_count == 1 else f"_{current_count:02d}"
    file_path = plotDir / f"{safe_name}{suffix}.png"
    fig.savefig(file_path, dpi=300, bbox_inches="tight")
    print("Saved plot:", file_path)
    plt.show()
    plt.close(fig)
    return file_path

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(randomState)
print("Project root:", projectRoot)
print("Plot directory:", plotDir)
print("Prediction directory:", predictionDir)


### Step 3 - Locate Dataset Automatically

Search for `metadata.csv` across several candidate locations and select the most likely dataset root automatically.


In [ ]:
search_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/input"),
    Path("/content"),
]

metadata_candidates = []
for root in search_roots:
    if root.exists():
        metadata_candidates.extend(root.rglob("metadata.csv"))

metadata_candidates = sorted({candidate.resolve() for candidate in metadata_candidates})
assert metadata_candidates, (
    "metadata.csv was not found automatically. Put the Brugada HUCA dataset inside the current "
    "workspace or inside /kaggle/input, then rerun this cell."
)

def dataset_score(meta_path):
    dataset_root = meta_path.parent
    hea_count = sum(1 for _ in dataset_root.rglob("*.hea"))
    dat_count = sum(1 for _ in dataset_root.rglob("*.dat"))
    return (hea_count + dat_count, hea_count, -len(str(meta_path)))

metaPath = max(metadata_candidates, key=dataset_score)
datasetRoot = metaPath.parent
metadata = pd.read_csv(metaPath)

requiredMetadataColumns = {"patient_id", "basal_pattern", "sudden_death", "brugada"}
missingMetadataColumns = requiredMetadataColumns.difference(metadata.columns)
assert not missingMetadataColumns, (
    f"Missing required metadata columns: {sorted(missingMetadataColumns)}"
)

metadata["brugada"] = pd.to_numeric(metadata["brugada"], errors="coerce")
classCounts = metadata["brugada"].value_counts(dropna=False).to_dict()
expectedTotalSubjects = 363
expectedBrugadaSubjects = 76
expectedNormalSubjects = 287

observedTotalSubjects = len(metadata)
observedBrugadaSubjects = int((metadata["brugada"] == 1).sum())
observedNormalSubjects = int((metadata["brugada"] == 0).sum())

datasetSpecChecks = [
    ("Total subjects", observedTotalSubjects, expectedTotalSubjects),
    ("Brugada subjects", observedBrugadaSubjects, expectedBrugadaSubjects),
    ("Normal subjects", observedNormalSubjects, expectedNormalSubjects),
]
datasetSpecIssues = []
for label, observed, expected in datasetSpecChecks:
    status = "OK" if observed == expected else "WARNING"
    print(f"{label}: observed={observed}, expected={expected} [{status}]")
    if observed != expected:
        datasetSpecIssues.append(f"{label} mismatch: observed {observed}, expected {expected}.")

if datasetSpecIssues:
    print("Dataset specification cross-check warnings:")
    for issue in datasetSpecIssues:
        print("-", issue)
    print("Execution will continue with the detected metadata values.")
else:
    print("Dataset specification cross-check passed.")

print("Metadata path:", metaPath)
print("Dataset root:", datasetRoot)
print("Metadata shape:", metadata.shape)
print("Metadata columns:", metadata.columns.tolist())
print("Class counts:", classCounts)
display(metadata.head())


## 2. EDA

This section checks the label distribution, ECG file availability, and a few initial signal examples.


### Step 4 - Inspect Label and Metadata Distribution

Review the Brugada vs Normal class balance and the distribution of key metadata fields. This step also highlights the class imbalance that must be handled during training and validation.


In [ ]:
targetColumn = "brugada"
assert targetColumn in metadata.columns, f"Expected '{targetColumn}' in metadata columns."

labelCounts = metadata[targetColumn].value_counts(dropna=False).sort_index()
positiveCount = int((metadata[targetColumn] == 1).sum())
negativeCount = int((metadata[targetColumn] == 0).sum())
imbalanceRatio = negativeCount / max(positiveCount, 1)
positiveRate = positiveCount / max(len(metadata), 1)

print(labelCounts)
print()
print(metadata[targetColumn].value_counts(normalize=True).rename("proportion"))
print()
print(f"Minority class (Brugada) count : {positiveCount}")
print(f"Majority class (Normal) count  : {negativeCount}")
print(f"Positive rate                  : {positiveRate:.4f}")
print(f"Imbalance ratio (Normal/Brugada): {imbalanceRatio:.2f}")
print(
    "Imbalance handling plan: stratified patient split, class weighting or scale_pos_weight, "
    "threshold tuning on validation data, and model ranking that emphasizes PR-AUC, F1, sensitivity, "
    "and Brugada detection instead of raw accuracy alone."
)

plt.figure(figsize=(5, 4))
sns.countplot(x=metadata[targetColumn])
plt.title("Label Distribution: 0 Normal, 1 Brugada")
plt.xlabel("Target")
plt.ylabel("Count")
save_plot("eda_label_distribution")

for column_name in ["basal_pattern", "sudden_death"]:
    if column_name in metadata.columns:
        plt.figure(figsize=(6, 4))
        sns.countplot(x=metadata[column_name].fillna("missing"))
        plt.title(f"Metadata Distribution: {column_name}")
        plt.xlabel(column_name)
        plt.ylabel("Count")
        plt.xticks(rotation=20)
        save_plot(f"eda_{column_name}_distribution")


### Step 5 - Index ECG Record Files

Build a WFDB file index so each `patient_id` can be matched to its ECG record.


In [ ]:
heaFiles = list(datasetRoot.rglob("*.hea"))
print("Total .hea files found:", len(heaFiles))

recordMap = {}
for hea in heaFiles:
    patient_key = Path(hea).stem
    recordMap[str(patient_key)] = str(Path(hea).with_suffix(""))

print("Indexed records:", len(recordMap))

missing_ids = [
    str(patient_id)
    for patient_id in metadata["patient_id"].astype(str)
    if str(patient_id) not in recordMap
]
print("Missing patient IDs in file index:", len(missing_ids))
if missing_ids:
    print("Example missing IDs:", missing_ids[:10])

expectedSamplingRate = 100
expectedDurationSeconds = 12
expectedSignalLength = expectedSamplingRate * expectedDurationSeconds

headerSummary = []
for patient_id, record_path in tqdm(sorted(recordMap.items()), total=len(recordMap), desc="Checking WFDB headers"):
    header = wfdb.rdheader(record_path)
    headerSummary.append(
        {
            "patient_id": patient_id,
            "fs": float(header.fs),
            "sig_len": int(header.sig_len),
            "n_sig": int(header.n_sig),
        }
    )

headerDf = pd.DataFrame(headerSummary)
assert not headerDf.empty, "No WFDB headers were indexed from the dataset root."

headerChecks = [
    ("Sampling rate", bool((headerDf["fs"] == expectedSamplingRate).all()), f"all records should be {expectedSamplingRate} Hz"),
    ("Signal length", bool((headerDf["sig_len"] == expectedSignalLength).all()), f"all records should be {expectedSignalLength} samples ({expectedDurationSeconds} seconds)"),
    ("Lead count", bool((headerDf["n_sig"] == len(standardLeads)).all()), f"all records should contain {len(standardLeads)} leads"),
]
headerIssues = []
for label, passed, expectation in headerChecks:
    status = "OK" if passed else "WARNING"
    print(f"{label} check: {status} ({expectation})")
    if not passed:
        headerIssues.append(f"{label} mismatch: {expectation}.")

if headerIssues:
    print("Recording specification cross-check warnings:")
    for issue in headerIssues:
        print("-", issue)
    print("Execution will continue with the detected recording properties.")
else:
    print("All records passed the recording specification cross-check.")

print("Expected sampling rate:", expectedSamplingRate, "Hz")
print("Expected signal length:", expectedSignalLength, "samples")
display(headerDf.head())


### Step 6 - Define ECG Loading Helper

Create a helper function to load ECG records and standardize the lead names.


In [ ]:
def standardizeLeadNames(sigNames):
    mapping = {}
    for name in sigNames:
        normalized = (
            str(name)
            .strip()
            .replace("AVR", "aVR")
            .replace("AVL", "aVL")
            .replace("AVF", "aVF")
        )
        mapping[name] = normalized
    return mapping

def loadEcgRecord(patientId):
    patientId = str(patientId)
    assert patientId in recordMap, f"Record for patient_id {patientId} not found."

    record = wfdb.rdrecord(recordMap[patientId])
    signal_df = pd.DataFrame(record.p_signal, columns=record.sig_name)
    signal_df = signal_df.rename(columns=standardizeLeadNames(signal_df.columns.tolist()))

    for lead_name in standardLeads:
        if lead_name not in signal_df.columns:
            raise ValueError(f"Lead {lead_name} not found for patient_id {patientId}")

    signal_df = signal_df[standardLeads].copy()
    return signal_df.values.astype(np.float32), standardLeads, int(record.fs)


### Step 7 - Visualize Example ECG Signals

Display example signals from one Normal patient and one Brugada patient for a quick visual check.


In [ ]:
normalId = metadata.loc[metadata[targetColumn] == 0, "patient_id"].iloc[0]
brugadaId = metadata.loc[metadata[targetColumn] == 1, "patient_id"].iloc[0]

def plotSelectedLeads(patientId, leadsToPlot=("V1", "V2", "V3", "II"), titlePrefix=""):
    X, leads, fs = loadEcgRecord(patientId)
    t = np.arange(len(X)) / fs

    plt.figure(figsize=(14, 8))
    for i, lead_name in enumerate(leadsToPlot, 1):
        lead_index = leads.index(lead_name)
        plt.subplot(len(leadsToPlot), 1, i)
        plt.plot(t, X[:, lead_index], linewidth=1.0)
        plt.title(f"{titlePrefix} patient_id={patientId} lead={lead_name}")
        plt.xlabel("Time (s)")
        plt.ylabel("mV")
    plt.tight_layout()
    save_plot(f"eda_{titlePrefix.lower()}_{patientId}_selected_leads")

plotSelectedLeads(normalId, titlePrefix="NORMAL")
plotSelectedLeads(brugadaId, titlePrefix="BRUGADA")


## 3. Preprocessing

This section cleans the signal, extracts the median beat, and builds both the feature-based and sequence-based representations.


### Step 8 - Define Signal Preprocessing Helpers

Create the filtering, robust normalization, R-peak detection, and median beat extraction helpers.

Important note: the dataset is loaded as raw ECG data, and the filtering and normalization below are notebook-level preprocessing choices added after loading the raw WFDB records.


In [ ]:
def bandpassFilter(x, fs, low=0.5, high=40.0, order=3):
    nyquist = 0.5 * fs
    low = max(low / nyquist, 1e-4)
    high = min(high / nyquist, 0.99)
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, x, axis=0)

def robustScale(x):
    median = np.median(x, axis=0, keepdims=True)
    mad = np.median(np.abs(x - median), axis=0, keepdims=True)
    mad = np.where(mad < 1e-6, 1.0, mad)
    return (x - median) / (1.4826 * mad)

def detectRPeaks(signal1d, fs):
    signal = signal1d.copy()
    if len(signal) >= 11:
        signal = savgol_filter(signal, 11, 3)
    signal = signal - np.median(signal)

    prominence = max(0.15, np.std(signal) * 0.5)
    distance = int(0.45 * fs)

    peaks, _ = find_peaks(signal, distance=distance, prominence=prominence)
    if len(peaks) == 0:
        peaks, _ = find_peaks(np.abs(signal), distance=distance, prominence=np.std(np.abs(signal)) * 0.3)
    return peaks

def extractMedianBeat(X, leads, fs):
    leadPriority = ["II", "V2", "V1", "V3", "I"]
    pre = int(0.25 * fs)
    post = int(0.45 * fs)

    peaks = np.array([], dtype=int)
    for refLead in leadPriority:
        if refLead in leads:
            refIndex = leads.index(refLead)
            peaks = detectRPeaks(X[:, refIndex], fs)
            peaks = peaks[(peaks > pre) & (peaks < len(X) - post)]
            if len(peaks) >= 2:
                break

    beats = []
    for peak in peaks:
        beat = X[peak - pre : peak + post, :]
        if beat.shape[0] == pre + post:
            beats.append(beat)

    if len(beats) == 0:
        center = len(X) // 2
        beat = X[max(0, center - pre) : min(len(X), center + post), :]
        padded = np.zeros((pre + post, X.shape[1]), dtype=np.float32)
        padded[: beat.shape[0], :] = beat
        return padded, pre

    medianBeat = np.median(np.stack(beats, axis=0), axis=0)
    return medianBeat.astype(np.float32), pre

def buildSequenceSample(patientId, focusLeads=standardLeads):
    X, leads, fs = loadEcgRecord(patientId)
    Xf = bandpassFilter(X, fs=fs, low=0.5, high=40.0, order=3)
    Xn = robustScale(Xf)
    medianBeat, _ = extractMedianBeat(Xn, leads, fs)
    lead_indices = [leads.index(lead_name) for lead_name in focusLeads]
    return medianBeat[:, lead_indices].astype(np.float32)


### Step 9 - Define Feature Engineering Helpers

Create the strip-level statistical features and the beat-level morphology features.


In [ ]:
def zeroCrossings(x):
    return int(np.sum(np.diff(np.signbit(x)).astype(int)))

def safeSlope(y):
    if len(y) < 2:
        return 0.0
    xs = np.arange(len(y))
    return float(np.polyfit(xs, y, 1)[0])

def extractStripFeatures(x, prefix):
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_ptp": float(np.ptp(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_q05": float(np.quantile(x, 0.05)),
        f"{prefix}_q25": float(np.quantile(x, 0.25)),
        f"{prefix}_q75": float(np.quantile(x, 0.75)),
        f"{prefix}_q95": float(np.quantile(x, 0.95)),
        f"{prefix}_rms": float(np.sqrt(np.mean(x**2))),
        f"{prefix}_abs_area": float(np.sum(np.abs(x))),
        f"{prefix}_energy": float(np.sum(x**2)),
        f"{prefix}_skew": float(skew(x)),
        f"{prefix}_kurtosis": float(kurtosis(x)),
        f"{prefix}_zero_cross": float(zeroCrossings(x)),
    }

def extractBeatFeatures(beat1d, rIndex, fs, prefix):
    qWindow = beat1d[max(0, rIndex - 8) : rIndex]
    sWindow = beat1d[rIndex : min(len(beat1d), rIndex + 12)]
    stWindow = beat1d[min(len(beat1d), rIndex + 6) : min(len(beat1d), rIndex + 16)]
    tWindow = beat1d[min(len(beat1d), rIndex + 18) : min(len(beat1d), rIndex + 35)]
    preWindow = beat1d[max(0, rIndex - 10) : max(1, rIndex - 2)]

    return {
        f"{prefix}_beat_r_amp": float(beat1d[rIndex]),
        f"{prefix}_beat_q_min": float(np.min(qWindow)) if len(qWindow) else 0.0,
        f"{prefix}_beat_s_min": float(np.min(sWindow)) if len(sWindow) else 0.0,
        f"{prefix}_beat_qrs_range": float(np.ptp(beat1d[max(0, rIndex - 6) : min(len(beat1d), rIndex + 10)])),
        f"{prefix}_beat_st_mean": float(np.mean(stWindow)) if len(stWindow) else 0.0,
        f"{prefix}_beat_st_max": float(np.max(stWindow)) if len(stWindow) else 0.0,
        f"{prefix}_beat_st_slope": float(safeSlope(stWindow)) if len(stWindow) else 0.0,
        f"{prefix}_beat_t_max": float(np.max(tWindow)) if len(tWindow) else 0.0,
        f"{prefix}_beat_t_mean": float(np.mean(tWindow)) if len(tWindow) else 0.0,
        f"{prefix}_beat_pre_mean": float(np.mean(preWindow)) if len(preWindow) else 0.0,
        f"{prefix}_beat_area": float(np.trapz(np.abs(beat1d))),
        f"{prefix}_beat_energy": float(np.sum(beat1d**2)),
    }

def buildFeatureRow(patientId):
    X, leads, fs = loadEcgRecord(patientId)

    Xf = bandpassFilter(X, fs=fs, low=0.5, high=40.0, order=3)
    Xn = robustScale(Xf)
    medianBeat, rIndex = extractMedianBeat(Xn, leads, fs)

    featureMap = {
        "patient_id": int(patientId),
        "fs": int(fs),
        "n_samples": int(Xn.shape[0]),
    }

    for leadIndex, leadName in enumerate(leads):
        featureMap.update(extractStripFeatures(Xn[:, leadIndex], prefix=f"{leadName}_strip"))

    for leadIndex, leadName in enumerate(leads):
        featureMap.update(extractBeatFeatures(medianBeat[:, leadIndex], rIndex=rIndex, fs=fs, prefix=f"{leadName}"))

    for leadName in ["V1", "V2", "V3"]:
        leadIndex = leads.index(leadName)
        segment = medianBeat[:, leadIndex]
        featureMap[f"{leadName}_post_r_100ms_mean"] = float(np.mean(segment[rIndex + 5 : rIndex + 15]))
        featureMap[f"{leadName}_post_r_150ms_mean"] = float(np.mean(segment[rIndex + 10 : rIndex + 20]))
        featureMap[f"{leadName}_post_r_200ms_mean"] = float(np.mean(segment[rIndex + 15 : rIndex + 25]))

    featureMap["V1_V2_st_mean_avg"] = float(np.mean([featureMap["V1_beat_st_mean"], featureMap["V2_beat_st_mean"]]))
    featureMap["V1_V3_st_mean_avg"] = float(np.mean([featureMap["V1_beat_st_mean"], featureMap["V3_beat_st_mean"]]))
    featureMap["V1_V2_qrs_avg"] = float(np.mean([featureMap["V1_beat_qrs_range"], featureMap["V2_beat_qrs_range"]]))
    featureMap["V1_V2_strip_energy_avg"] = float(np.mean([featureMap["V1_strip_energy"], featureMap["V2_strip_energy"]]))
    featureMap["V1_minus_V2_st_mean"] = float(featureMap["V1_beat_st_mean"] - featureMap["V2_beat_st_mean"])
    featureMap["V2_minus_V3_st_mean"] = float(featureMap["V2_beat_st_mean"] - featureMap["V3_beat_st_mean"])

    for lead_a, lead_b in [("V1", "V2"), ("V2", "V3"), ("I", "II"), ("II", "V1")]:
        idx_a, idx_b = leads.index(lead_a), leads.index(lead_b)
        corr = np.corrcoef(Xn[:, idx_a], Xn[:, idx_b])[0, 1]
        featureMap[f"corr_{lead_a}_{lead_b}"] = float(np.nan_to_num(corr))

    return featureMap


### Step 10 - Build Feature Table

Run preprocessing for every patient and combine the outputs into one feature table.


In [ ]:
featureRows = []

for row in tqdm(metadata.itertuples(index=False), total=len(metadata)):
    patientId = getattr(row, "patient_id")
    try:
        featureMap = buildFeatureRow(patientId)
        featureMap["target"] = int(getattr(row, targetColumn))
        if "basal_pattern" in metadata.columns:
            featureMap["basal_pattern_meta"] = getattr(row, "basal_pattern", np.nan)
        if "sudden_death" in metadata.columns:
            featureMap["sudden_death_meta"] = getattr(row, "sudden_death", np.nan)
        featureRows.append(featureMap)
    except Exception as exc:
        print(f"Skipping patient_id={patientId} because of preprocessing error: {exc}")

featuresDf = pd.DataFrame(featureRows)
print("Feature table shape:", featuresDf.shape)
display(featuresDf.head())

missingSummary = featuresDf.isna().mean().sort_values(ascending=False)
display(missingSummary.head(20))

plt.figure(figsize=(6, 4))
sns.histplot(missingSummary.values, bins=20)
plt.title("Missing Ratio Across Engineered Features")
plt.xlabel("Missing ratio")
plt.ylabel("Count")
save_plot("preprocessing_missing_ratio_histogram")


## 4. Test Split

This section builds one shared patient split for all models so the comparison remains fair.


### Step 11 - Create the Shared Patient Split

Split the data into train, validation, and test sets at the patient level with stratification, then prepare the tabular and sequence inputs. Stratification is required here because the Brugada class is the minority class.


In [ ]:
benchmarkDf = featuresDf.copy()
benchmarkDf["patient_id"] = pd.to_numeric(benchmarkDf["patient_id"], errors="coerce")
benchmarkDf["target"] = pd.to_numeric(benchmarkDf["target"], errors="coerce")
benchmarkDf = benchmarkDf.dropna(subset=["patient_id", "target"]).copy()
benchmarkDf = benchmarkDf[benchmarkDf["target"].isin([0, 1])].copy()
benchmarkDf["patient_id"] = benchmarkDf["patient_id"].astype(int)
benchmarkDf["target"] = benchmarkDf["target"].astype(int)
benchmarkDf = benchmarkDf.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)

sequenceCache = {}
validPatients = []
for row in tqdm(benchmarkDf[["patient_id", "target"]].itertuples(index=False), total=len(benchmarkDf)):
    pid = int(row.patient_id)
    try:
        sequenceCache[pid] = buildSequenceSample(pid, focusLeads=standardLeads)
        validPatients.append({"patient_id": pid, "target": int(row.target)})
    except Exception as exc:
        print(f"Skipping patient_id={pid} because median beat extraction failed: {exc}")

benchmarkPatientDf = pd.DataFrame(validPatients).sort_values("patient_id").reset_index(drop=True)
benchmarkDf = benchmarkDf[benchmarkDf["patient_id"].isin(benchmarkPatientDf["patient_id"])].copy()
benchmarkDf = benchmarkDf.sort_values("patient_id").reset_index(drop=True)

trainDf, testDf = train_test_split(
    benchmarkPatientDf,
    test_size=0.20,
    stratify=benchmarkPatientDf["target"],
    random_state=randomState,
)
trainDf, valDf = train_test_split(
    trainDf,
    test_size=0.20,
    stratify=trainDf["target"],
    random_state=randomState,
)

splitDf = pd.concat(
    [
        trainDf.assign(split="train"),
        valDf.assign(split="validation"),
        testDf.assign(split="test"),
    ],
    axis=0,
).reset_index(drop=True)
splitFile = predictionDir / "shared_patient_split.csv"
splitDf.to_csv(splitFile, index=False)

trainIds = trainDf["patient_id"].tolist()
valIds = valDf["patient_id"].tolist()
testIds = testDf["patient_id"].tolist()

labelByPatient = benchmarkPatientDf.set_index("patient_id")["target"].to_dict()
featureColumns = [column for column in benchmarkDf.columns if column not in ["patient_id", "target"]]
featureByPatient = benchmarkDf.set_index("patient_id")[featureColumns].sort_index()

def getTabularSplit(patientIds):
    x_df = featureByPatient.loc[patientIds].copy()
    y = np.array([labelByPatient[int(pid)] for pid in patientIds], dtype=int)
    return x_df, y

def getSequenceSplit(patientIds):
    X = np.stack([sequenceCache[int(pid)] for pid in patientIds]).astype(np.float32)
    y = np.array([labelByPatient[int(pid)] for pid in patientIds], dtype=int)
    return X, y

xTrainTabDf, yTrainTab = getTabularSplit(trainIds)
xValTabDf, yValTab = getTabularSplit(valIds)
xTestTabDf, yTestTab = getTabularSplit(testIds)

benchmarkImputer = SimpleImputer(strategy="median")
xTrainTab = benchmarkImputer.fit_transform(xTrainTabDf)
xValTab = benchmarkImputer.transform(xValTabDf)
xTestTab = benchmarkImputer.transform(xTestTabDf)

benchmarkScaler = StandardScaler()
xTrainTabScaled = benchmarkScaler.fit_transform(xTrainTab)
xValTabScaled = benchmarkScaler.transform(xValTab)
xTestTabScaled = benchmarkScaler.transform(xTestTab)

xTrainSeq, yTrainSeq = getSequenceSplit(trainIds)
xValSeq, yValSeq = getSequenceSplit(valIds)
xTestSeq, yTestSeq = getSequenceSplit(testIds)

print("Shared split file:", splitFile)
print("Benchmark patient count:", len(benchmarkPatientDf))
print("Train / Validation / Test:", len(trainIds), len(valIds), len(testIds))
print("Train labels:", np.unique(yTrainTab, return_counts=True))
print("Validation labels:", np.unique(yValTab, return_counts=True))
print("Test labels:", np.unique(yTestTab, return_counts=True))
print("Tabular train shape:", xTrainTab.shape)
print("Sequence train shape:", xTrainSeq.shape)


### Step 12 - Visualize the Shared Split

Display the class distribution in each split to confirm that the split remains balanced.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=splitDf, x="split", hue="target")
plt.title("Shared Patient Split by Class")
plt.xlabel("Split")
plt.ylabel("Patient count")
save_plot("test_split_shared_patient_distribution")

display(splitDf.head())


## 5. Train Models

All models below use the same patient split. This section is divided into feature-based, deep learning, and transfer learning strategies. Because the dataset is imbalanced, the training setup uses imbalance-aware weighting, threshold tuning, and validation metrics that do not rely on accuracy alone.


### 5.1 Feature-Based Models

The models in this section use manually engineered ECG features.


### Step 13 - Define Evaluation and Export Helpers

Create the helper functions for threshold tuning, metric evaluation, and prediction export. These helpers are designed to keep the benchmark imbalance-aware by focusing on PR-AUC, F1, sensitivity, balanced accuracy, and class-level Brugada detection.


In [ ]:
def specificityScore(yTrue, yPred):
    yTrue = np.asarray(yTrue).astype(int)
    yPred = np.asarray(yPred).astype(int)
    matrix = confusion_matrix(yTrue, yPred, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def sensitivityScore(yTrue, yPred):
    yTrue = np.asarray(yTrue).astype(int)
    yPred = np.asarray(yPred).astype(int)
    matrix = confusion_matrix(yTrue, yPred, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def tuneThreshold(yTrue, yProb):
    yTrue = np.asarray(yTrue).astype(int)
    yProb = np.asarray(yProb).astype(float)

    precisionValues, recallValues, thresholdValues = precision_recall_curve(yTrue, yProb)
    if len(thresholdValues) == 0:
        defaultThreshold = 0.5
        defaultF1 = f1_score(yTrue, (yProb >= defaultThreshold).astype(int))
        return float(defaultThreshold), float(defaultF1)

    precisionValues = precisionValues[:-1]
    recallValues = recallValues[:-1]
    f1Values = 2 * precisionValues * recallValues / np.clip(precisionValues + recallValues, 1e-8, None)
    bestIndex = int(np.nanargmax(f1Values))
    return float(thresholdValues[bestIndex]), float(f1Values[bestIndex])

def classCountReport(yTrue, yPred, modelName):
    yTrue = np.asarray(yTrue).astype(int)
    yPred = np.asarray(yPred).astype(int)
    rows = []
    for cls, className in [(0, "Normal"), (1, "Brugada")]:
        actualCount = int(np.sum(yTrue == cls))
        predictedCount = int(np.sum(yPred == cls))
        correctCount = int(np.sum((yTrue == cls) & (yPred == cls)))
        rows.append(
            {
                "Model": modelName,
                "Class": f"{className} ({cls})",
                "Actual Count": actualCount,
                "Predicted Count": predictedCount,
                "Correctly Predicted": correctCount,
                "Recall per Class": correctCount / actualCount if actualCount > 0 else np.nan,
                "Precision on Predicted Class": correctCount / predictedCount if predictedCount > 0 else np.nan,
            }
        )
    rows.append(
        {
            "Model": modelName,
            "Class": "TOTAL",
            "Actual Count": int(len(yTrue)),
            "Predicted Count": int(len(yPred)),
            "Correctly Predicted": int(np.sum(yTrue == yPred)),
            "Recall per Class": float(np.mean(yTrue == yPred)),
            "Precision on Predicted Class": np.nan,
        }
    )
    return pd.DataFrame(rows)

def evaluateBinaryPredictions(modelName, family, patientIds, yTrue, yProb, threshold):
    yTrue = np.asarray(yTrue).astype(int)
    yProb = np.asarray(yProb).astype(float)
    yPred = (yProb >= threshold).astype(int)

    predictionDf = pd.DataFrame(
        {
            "patient_id": [int(pid) for pid in patientIds],
            "target": yTrue,
            "pred_proba": yProb,
            "pred_label": yPred,
        }
    )
    predictionFile = predictionDir / f"{sanitize_name(modelName)}_predictions.csv"
    predictionDf.to_csv(predictionFile, index=False)

    metrics = {
        "Model": modelName,
        "Family": family,
        "Threshold": float(threshold),
        "AUC": float(roc_auc_score(yTrue, yProb)) if len(np.unique(yTrue)) > 1 else np.nan,
        "PR AUC": float(average_precision_score(yTrue, yProb)) if len(np.unique(yTrue)) > 1 else np.nan,
        "F1": float(f1_score(yTrue, yPred)),
        "Accuracy": float(accuracy_score(yTrue, yPred)),
        "Balanced Accuracy": float(balanced_accuracy_score(yTrue, yPred)),
        "Sensitivity": float(sensitivityScore(yTrue, yPred)),
        "Specificity": float(specificityScore(yTrue, yPred)),
        "Correctly Predicted": int(np.sum(yTrue == yPred)),
        "Total Samples": int(len(yTrue)),
        "Correct Normal": int(np.sum((yTrue == 0) & (yPred == 0))),
        "Correct Brugada": int(np.sum((yTrue == 1) & (yPred == 1))),
        "Prediction CSV": str(predictionFile),
    }
    countDf = classCountReport(yTrue, yPred, modelName)
    return metrics, predictionDf, countDf

def makeClassWeightDict(y):
    y = np.asarray(y).astype(int)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(cls): float(weight) for cls, weight in zip(classes, weights)}

def makeSampleWeightVector(y):
    classWeight = makeClassWeightDict(y)
    y = np.asarray(y).astype(int)
    return np.array([classWeight[int(label)] for label in y], dtype=float)

def makeTrainingCallbacks(monitor="val_auc", mode="max", patience=12):
    return [
        callbacks.EarlyStopping(
            monitor=monitor,
            mode=mode,
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        )
    ]

def historyToDict(history):
    return {key: [float(v) for v in values] for key, values in history.history.items()}

benchmarkResults = []
benchmarkCountReports = []
benchmarkPredictionTables = {}
benchmarkHistories = {}


### Step 14 - Train Feature-Based Models

Train `XGBoost`, `Random Forest`, `SVM RBF`, and `AdaBoost` on the same feature table. This step uses imbalance-aware settings such as `scale_pos_weight`, balanced class weights, weighted AdaBoost fitting, and validation-based threshold tuning.


In [ ]:
scalePosWeightBenchmark = (len(yTrainTab) - yTrainTab.sum()) / max(yTrainTab.sum(), 1)
trainClassWeightMap = makeClassWeightDict(yTrainTab)
trainSampleWeightVector = makeSampleWeightVector(yTrainTab)

print("Training class weights:", trainClassWeightMap)
print("scale_pos_weight for XGBoost:", float(scalePosWeightBenchmark))

classicalModels = {
    "XGBoost Features": XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.3,
        reg_lambda=1.5,
        min_child_weight=2,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=randomState,
        scale_pos_weight=scalePosWeightBenchmark,
        tree_method="hist",
    ),
    "Random Forest Features": RandomForestClassifier(
        n_estimators=600,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=randomState,
        n_jobs=-1,
    ),
    "SVM RBF Features": SVC(
        kernel="rbf",
        C=2.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=randomState,
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=350,
        learning_rate=0.05,
        random_state=randomState,
    ),
}

for modelName, model in classicalModels.items():
    print()
    print(f"Training {modelName} ...")

    if "SVM" in modelName:
        xTrainInput, xValInput, xTestInput = xTrainTabScaled, xValTabScaled, xTestTabScaled
    else:
        xTrainInput, xValInput, xTestInput = xTrainTab, xValTab, xTestTab

    if modelName == "AdaBoost":
        model.fit(xTrainInput, yTrainTab, sample_weight=trainSampleWeightVector)
    else:
        model.fit(xTrainInput, yTrainTab)

    valProba = model.predict_proba(xValInput)[:, 1].astype(float)
    bestThreshold, bestValF1 = tuneThreshold(yValTab, valProba)
    testProba = model.predict_proba(xTestInput)[:, 1].astype(float)

    modelMetrics, predictionDf, countDf = evaluateBinaryPredictions(
        modelName=modelName,
        family="Feature-Based",
        patientIds=testIds,
        yTrue=yTestTab,
        yProb=testProba,
        threshold=bestThreshold,
    )
    modelMetrics["Validation F1"] = float(bestValF1)

    benchmarkResults.append(modelMetrics)
    benchmarkCountReports.append(countDf)
    benchmarkPredictionTables[modelName] = predictionDf

    print(
        f"{modelName} | PR-AUC={modelMetrics['PR AUC']:.4f} | F1={modelMetrics['F1']:.4f} | "
        f"Sensitivity={modelMetrics['Sensitivity']:.4f} | Correct Brugada={modelMetrics['Correct Brugada']}"
    )
    print("Prediction CSV:", modelMetrics["Prediction CSV"])


### 5.2 Deep Learning End-to-End Models

The models in this section learn directly from the 12-lead median beat ECG representation.


### Step 15 - Define Deep Learning Architectures

Create the `1D CNN`, `ResNet 1D`, `CNN + BiGRU`, and `Transformer Encoder` architectures.


In [ ]:
def residualBlock(x, filters, kernelSize=3, stride=1, dropoutRate=0.10):
    shortcut = x
    if stride != 1 or int(x.shape[-1]) != filters:
        shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    y = layers.Conv1D(filters, kernelSize, strides=stride, padding="same")(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv1D(filters, kernelSize, padding="same")(y)
    y = layers.BatchNormalization()(y)
    if dropoutRate > 0:
        y = layers.SpatialDropout1D(dropoutRate)(y)

    out = layers.Add()([shortcut, y])
    out = layers.Activation("relu")(out)
    return out

def buildCnn1d(inputShape, learningRate=1e-3):
    inp = layers.Input(shape=inputShape)
    x = layers.Conv1D(32, 7, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="cnn_1d")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model

def buildResnet1d(inputShape, learningRate=1e-3):
    inp = layers.Input(shape=inputShape)
    x = layers.Conv1D(32, 7, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = residualBlock(x, 32, stride=1, dropoutRate=0.05)
    x = residualBlock(x, 64, stride=2, dropoutRate=0.10)
    x = residualBlock(x, 128, stride=2, dropoutRate=0.10)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="resnet_1d")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model

def buildCnnBigru(inputShape, learningRate=1e-3):
    inp = layers.Input(shape=inputShape)
    x = layers.Conv1D(32, 5, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv1D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.GRU(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.GRU(32))(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="cnn_bigru")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model

def transformerBlock(x, numHeads=4, ffDim=128, dropoutRate=0.10):
    attn = layers.MultiHeadAttention(
        num_heads=numHeads,
        key_dim=max(8, int(x.shape[-1]) // numHeads),
        dropout=dropoutRate,
    )(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ffDim, activation="relu")(x)
    ff = layers.Dropout(dropoutRate)(ff)
    ff = layers.Dense(int(x.shape[-1]))(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x

def buildTransformerModel(inputShape, learningRate=8e-4):
    inp = layers.Input(shape=inputShape)
    modelDim = 64
    x = layers.Dense(modelDim)(inp)
    positionIndex = tf.range(start=0, limit=inputShape[0], delta=1)
    positionEmbedding = layers.Embedding(input_dim=inputShape[0], output_dim=modelDim)(positionIndex)
    x = layers.Add()([x, positionEmbedding])
    x = transformerBlock(x, numHeads=4, ffDim=128, dropoutRate=0.10)
    x = transformerBlock(x, numHeads=4, ffDim=128, dropoutRate=0.10)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="transformer_1d")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model



### Step 16 - Define the Deep Training Routine

Create one shared training routine so every deep learning model follows a consistent workflow.


In [ ]:
def trainDeepClassifier(modelName, modelBuilder, xTrain, yTrain, xVal, yVal, xTest, yTest, testPatientIds, learningRate=1e-3, epochs=60, batchSize=16):
    tf.keras.backend.clear_session()
    gc.collect()
    set_global_seed(randomState)

    xTrain = np.asarray(xTrain).astype(np.float32)
    xVal = np.asarray(xVal).astype(np.float32)
    xTest = np.asarray(xTest).astype(np.float32)
    yTrain = np.asarray(yTrain).astype(int).ravel()
    yVal = np.asarray(yVal).astype(int).ravel()
    yTest = np.asarray(yTest).astype(int).ravel()

    model = modelBuilder(inputShape=xTrain.shape[1:], learningRate=learningRate)
    classWeight = makeClassWeightDict(yTrain)

    history = model.fit(
        xTrain,
        yTrain,
        validation_data=(xVal, yVal),
        epochs=epochs,
        batch_size=batchSize,
        callbacks=makeTrainingCallbacks(monitor="val_auc", mode="max", patience=10),
        class_weight=classWeight,
        verbose=1,
    )

    valProba = model.predict(xVal, verbose=0).ravel()
    bestThreshold, bestValF1 = tuneThreshold(yVal, valProba)
    testProba = model.predict(xTest, verbose=0).ravel()

    modelMetrics, predictionDf, countDf = evaluateBinaryPredictions(
        modelName=modelName,
        family="Deep Learning End-to-End",
        patientIds=testPatientIds,
        yTrue=yTest,
        yProb=testProba,
        threshold=bestThreshold,
    )
    modelMetrics["Validation F1"] = float(bestValF1)
    modelMetrics["Epochs Trained"] = int(len(history.history.get("loss", [])))

    historyDict = historyToDict(history)

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return modelMetrics, predictionDf, countDf, historyDict


### Step 17 - Train Deep Learning Models

Train all four deep learning models on the same median beat split.


In [ ]:
deepModelBuilders = [
    ("1D CNN Median Beat", buildCnn1d, 1e-3),
    ("ResNet 1D Median Beat", buildResnet1d, 1e-3),
    ("CNN + BiGRU Median Beat", buildCnnBigru, 1e-3),
    ("Transformer Encoder Median Beat", buildTransformerModel, 8e-4),
]

for modelName, modelBuilder, learningRate in deepModelBuilders:
    print(f"\nTraining {modelName} ...")
    modelMetrics, predictionDf, countDf, historyDict = trainDeepClassifier(
        modelName=modelName,
        modelBuilder=modelBuilder,
        xTrain=xTrainSeq,
        yTrain=yTrainSeq,
        xVal=xValSeq,
        yVal=yValSeq,
        xTest=xTestSeq,
        yTest=yTestSeq,
        testPatientIds=testIds,
        learningRate=learningRate,
        epochs=60,
        batchSize=16,
    )
    benchmarkResults.append(modelMetrics)
    benchmarkCountReports.append(countDf)
    benchmarkPredictionTables[modelName] = predictionDf
    benchmarkHistories[modelName] = historyDict

    print(
        f"{modelName} | AUC={modelMetrics['AUC']:.4f} | F1={modelMetrics['F1']:.4f} | "
        f"Correct={modelMetrics['Correctly Predicted']}/{modelMetrics['Total Samples']}"
    )
    print("Prediction CSV:", modelMetrics["Prediction CSV"])


### 5.3 Transfer Learning and Advanced Models

This section focuses on advanced strategies that help when Brugada data is limited.


### Step 18 - Define In-Domain Transfer Learning Helpers

Create the encoder, denoising autoencoder, and classifier used by the in-domain transfer learning pipeline.


In [ ]:
def buildTransferEncoder(inputShape):
    inp = layers.Input(shape=inputShape)
    x = layers.Conv1D(32, 5, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2, padding="same")(x)
    x = residualBlock(x, 64, stride=1, dropoutRate=0.05)
    x = layers.MaxPooling1D(2, padding="same")(x)
    x = residualBlock(x, 128, stride=1, dropoutRate=0.10)
    return models.Model(inp, x, name="transfer_encoder")

def buildDenoisingAutoencoder(inputShape):
    encoder = buildTransferEncoder(inputShape)
    inp = layers.Input(shape=inputShape)
    x = layers.GaussianNoise(0.05)(inp)
    x = encoder(x)
    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
    x = layers.Conv1D(inputShape[-1], 1, padding="same")(x)

    currentLen = int(x.shape[1])
    targetLen = inputShape[0]
    if currentLen > targetLen:
        cropTotal = currentLen - targetLen
        cropLeft = cropTotal // 2
        cropRight = cropTotal - cropLeft
        x = layers.Cropping1D((cropLeft, cropRight))(x)
    elif currentLen < targetLen:
        padTotal = targetLen - currentLen
        padLeft = padTotal // 2
        padRight = padTotal - padLeft
        x = layers.ZeroPadding1D((padLeft, padRight))(x)

    autoencoder = models.Model(inp, x, name="denoising_autoencoder")
    autoencoder.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss="mse", jit_compile=False)
    return encoder, autoencoder

def buildTransferClassifier(encoder, inputShape, learningRate=3e-4):
    inp = layers.Input(shape=inputShape)
    x = encoder(inp)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="transfer_classifier")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model

def trainTransferLearningModel(modelName, xTrain, yTrain, xVal, yVal, xTest, yTest, testPatientIds, pretrainEpochs=40, finetuneEpochs=60, batchSize=16):
    tf.keras.backend.clear_session()
    gc.collect()
    set_global_seed(randomState)

    xTrain = np.asarray(xTrain).astype(np.float32)
    xVal = np.asarray(xVal).astype(np.float32)
    xTest = np.asarray(xTest).astype(np.float32)
    yTrain = np.asarray(yTrain).astype(int).ravel()
    yVal = np.asarray(yVal).astype(int).ravel()
    yTest = np.asarray(yTest).astype(int).ravel()

    encoder, autoencoder = buildDenoisingAutoencoder(inputShape=xTrain.shape[1:])
    pretrainHistory = autoencoder.fit(
        xTrain,
        xTrain,
        validation_data=(xVal, xVal),
        epochs=pretrainEpochs,
        batch_size=batchSize,
        callbacks=makeTrainingCallbacks(monitor="val_loss", mode="min", patience=8),
        verbose=1,
    )

    classifier = buildTransferClassifier(
        encoder=encoder,
        inputShape=xTrain.shape[1:],
        learningRate=3e-4,
    )
    classWeight = makeClassWeightDict(yTrain)
    finetuneHistory = classifier.fit(
        xTrain,
        yTrain,
        validation_data=(xVal, yVal),
        epochs=finetuneEpochs,
        batch_size=batchSize,
        callbacks=makeTrainingCallbacks(monitor="val_auc", mode="max", patience=10),
        class_weight=classWeight,
        verbose=1,
    )

    valProba = classifier.predict(xVal, verbose=0).ravel()
    bestThreshold, bestValF1 = tuneThreshold(yVal, valProba)
    testProba = classifier.predict(xTest, verbose=0).ravel()

    modelMetrics, predictionDf, countDf = evaluateBinaryPredictions(
        modelName=modelName,
        family="Transfer Learning / Advanced",
        patientIds=testPatientIds,
        yTrue=yTest,
        yProb=testProba,
        threshold=bestThreshold,
    )
    modelMetrics["Validation F1"] = float(bestValF1)
    modelMetrics["Pretrain Epochs"] = int(len(pretrainHistory.history.get("loss", [])))
    modelMetrics["Epochs Trained"] = int(len(finetuneHistory.history.get("loss", [])))

    historyDict = {
        "pretrain": historyToDict(pretrainHistory),
        "finetune": historyToDict(finetuneHistory),
    }

    del autoencoder
    del classifier
    tf.keras.backend.clear_session()
    gc.collect()
    return modelMetrics, predictionDf, countDf, historyDict



### Step 19 - Define VICReg Self-Supervised Helpers

Create the `VICReg` components for label-free pretraining on the same Brugada HUCA data.


In [ ]:
def buildVicRegEncoder(inputShape, embeddingDim=64):
    inp = layers.Input(shape=inputShape)
    x = layers.Conv1D(32, 5, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = residualBlock(x, 64, stride=2, dropoutRate=0.05)
    x = residualBlock(x, 128, stride=2, dropoutRate=0.10)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(embeddingDim)(x)
    return models.Model(inp, out, name="vicreg_encoder")

def augmentEcgBatch(batch):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), stddev=0.03)
    scale = tf.random.uniform((tf.shape(batch)[0], 1, 1), minval=0.90, maxval=1.10)
    mask = tf.cast(
        tf.random.uniform((tf.shape(batch)[0], tf.shape(batch)[1], 1), minval=0.0, maxval=1.0) > 0.08,
        tf.float32,
    )
    return (batch * scale + noise) * mask

def vicregLoss(z1, z2, simCoeff=25.0, stdCoeff=25.0, covCoeff=1.0):
    z1 = tf.cast(z1, tf.float32)
    z2 = tf.cast(z2, tf.float32)

    reprLoss = tf.reduce_mean(tf.square(z1 - z2))

    z1Centered = z1 - tf.reduce_mean(z1, axis=0)
    z2Centered = z2 - tf.reduce_mean(z2, axis=0)

    stdZ1 = tf.sqrt(tf.math.reduce_variance(z1Centered, axis=0) + 1e-4)
    stdZ2 = tf.sqrt(tf.math.reduce_variance(z2Centered, axis=0) + 1e-4)
    stdLoss = tf.reduce_mean(tf.nn.relu(1.0 - stdZ1)) + tf.reduce_mean(tf.nn.relu(1.0 - stdZ2))

    batchSize = tf.cast(tf.shape(z1Centered)[0], tf.float32)
    covZ1 = tf.matmul(z1Centered, z1Centered, transpose_a=True) / tf.maximum(batchSize - 1.0, 1.0)
    covZ2 = tf.matmul(z2Centered, z2Centered, transpose_a=True) / tf.maximum(batchSize - 1.0, 1.0)

    dim = tf.shape(covZ1)[0]
    offDiagMask = tf.ones((dim, dim), dtype=tf.float32) - tf.eye(dim, dtype=tf.float32)
    covLoss = tf.reduce_mean(tf.square(covZ1 * offDiagMask)) + tf.reduce_mean(tf.square(covZ2 * offDiagMask))

    return simCoeff * reprLoss + stdCoeff * stdLoss + covCoeff * covLoss

def pretrainVicRegEncoder(xTrain, xVal, epochs=30, batchSize=16, learningRate=1e-3, patience=6):
    set_global_seed(randomState)
    encoder = buildVicRegEncoder(xTrain.shape[1:])
    optimizer = optimizers.Adam(learning_rate=learningRate)

    trainDs = (
        tf.data.Dataset.from_tensor_slices(np.asarray(xTrain).astype(np.float32))
        .shuffle(len(xTrain), seed=randomState, reshuffle_each_iteration=True)
        .batch(batchSize)
    )
    valDs = tf.data.Dataset.from_tensor_slices(np.asarray(xVal).astype(np.float32)).batch(batchSize)

    history = {"loss": [], "val_loss": []}
    bestWeights = encoder.get_weights()
    bestVal = np.inf
    wait = 0

    for epoch in range(epochs):
        trainLosses = []
        for batch in trainDs:
            with tf.GradientTape() as tape:
                view1 = augmentEcgBatch(batch)
                view2 = augmentEcgBatch(batch)
                z1 = encoder(view1, training=True)
                z2 = encoder(view2, training=True)
                loss = vicregLoss(z1, z2)
            gradients = tape.gradient(loss, encoder.trainable_weights)
            optimizer.apply_gradients(zip(gradients, encoder.trainable_weights))
            trainLosses.append(float(loss.numpy()))

        valLosses = []
        for batch in valDs:
            view1 = augmentEcgBatch(batch)
            view2 = augmentEcgBatch(batch)
            z1 = encoder(view1, training=False)
            z2 = encoder(view2, training=False)
            valLosses.append(float(vicregLoss(z1, z2).numpy()))

        epochTrainLoss = float(np.mean(trainLosses))
        epochValLoss = float(np.mean(valLosses))
        history["loss"].append(epochTrainLoss)
        history["val_loss"].append(epochValLoss)
        print(f"VICReg pretrain epoch {epoch + 1:02d} | loss={epochTrainLoss:.4f} | val_loss={epochValLoss:.4f}")

        if epochValLoss < bestVal - 1e-4:
            bestVal = epochValLoss
            bestWeights = encoder.get_weights()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("VICReg pretraining early stop.")
                break

    encoder.set_weights(bestWeights)
    return encoder, history

def buildVicRegClassifier(encoder, inputShape, learningRate=3e-4):
    inp = layers.Input(shape=inputShape)
    x = encoder(inp)
    x = layers.LayerNormalization()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out, name="vicreg_classifier")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learningRate),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
        jit_compile=False,
    )
    return model

def trainVicRegModel(modelName, xTrain, yTrain, xVal, yVal, xTest, yTest, testPatientIds, pretrainEpochs=30, finetuneEpochs=50, batchSize=16):
    tf.keras.backend.clear_session()
    gc.collect()
    set_global_seed(randomState)

    xTrain = np.asarray(xTrain).astype(np.float32)
    xVal = np.asarray(xVal).astype(np.float32)
    xTest = np.asarray(xTest).astype(np.float32)
    yTrain = np.asarray(yTrain).astype(int).ravel()
    yVal = np.asarray(yVal).astype(int).ravel()
    yTest = np.asarray(yTest).astype(int).ravel()

    encoder, pretrainHistory = pretrainVicRegEncoder(
        xTrain=xTrain,
        xVal=xVal,
        epochs=pretrainEpochs,
        batchSize=batchSize,
        learningRate=1e-3,
        patience=6,
    )
    classifier = buildVicRegClassifier(encoder=encoder, inputShape=xTrain.shape[1:], learningRate=3e-4)
    classWeight = makeClassWeightDict(yTrain)
    finetuneHistory = classifier.fit(
        xTrain,
        yTrain,
        validation_data=(xVal, yVal),
        epochs=finetuneEpochs,
        batch_size=batchSize,
        callbacks=makeTrainingCallbacks(monitor="val_auc", mode="max", patience=8),
        class_weight=classWeight,
        verbose=1,
    )

    valProba = classifier.predict(xVal, verbose=0).ravel()
    bestThreshold, bestValF1 = tuneThreshold(yVal, valProba)
    testProba = classifier.predict(xTest, verbose=0).ravel()

    modelMetrics, predictionDf, countDf = evaluateBinaryPredictions(
        modelName=modelName,
        family="Transfer Learning / Advanced",
        patientIds=testPatientIds,
        yTrue=yTest,
        yProb=testProba,
        threshold=bestThreshold,
    )
    modelMetrics["Validation F1"] = float(bestValF1)
    modelMetrics["Pretrain Epochs"] = int(len(pretrainHistory["loss"]))
    modelMetrics["Epochs Trained"] = int(len(finetuneHistory.history.get("loss", [])))

    historyDict = {
        "pretrain": pretrainHistory,
        "finetune": historyToDict(finetuneHistory),
    }

    del classifier
    tf.keras.backend.clear_session()
    gc.collect()
    return modelMetrics, predictionDf, countDf, historyDict



### Step 20 - Define Echo State Network Helper

Create the reservoir transformation and classifier pipeline for the `ESN` model.


In [ ]:
def esnTransform(X, nReservoir=160, spectralRadius=0.90, leakRate=0.35, seed=42):
    rng = np.random.default_rng(seed)
    X = np.asarray(X).astype(np.float32)
    nInputs = X.shape[-1]
    win = rng.uniform(-0.5, 0.5, size=(nInputs, nReservoir)).astype(np.float32)
    w = rng.uniform(-0.5, 0.5, size=(nReservoir, nReservoir)).astype(np.float32)
    eigenvalues = np.linalg.eigvals(w)
    spectralNorm = np.max(np.abs(eigenvalues))
    w *= spectralRadius / max(float(spectralNorm), 1e-6)
    bias = rng.uniform(-0.1, 0.1, size=(nReservoir,)).astype(np.float32)

    featureRows = []
    for sample in X:
        state = np.zeros(nReservoir, dtype=np.float32)
        stateHistory = []
        for timeIndex in range(sample.shape[0]):
            inputVector = sample[timeIndex]
            preActivation = inputVector @ win + state @ w + bias
            state = (1.0 - leakRate) * state + leakRate * np.tanh(preActivation)
            stateHistory.append(state.copy())
        stateHistory = np.stack(stateHistory, axis=0)
        featureRows.append(
            np.concatenate(
                [
                    stateHistory.mean(axis=0),
                    stateHistory.std(axis=0),
                    stateHistory[-1],
                ],
                axis=0,
            )
        )
    return np.stack(featureRows).astype(np.float32)

def trainEsnModel(modelName, xTrain, yTrain, xVal, yVal, xTest, yTest, testPatientIds):
    xTrainReservoir = esnTransform(xTrain, seed=randomState)
    xValReservoir = esnTransform(xVal, seed=randomState)
    xTestReservoir = esnTransform(xTest, seed=randomState)

    scaler = StandardScaler()
    xTrainReservoir = scaler.fit_transform(xTrainReservoir)
    xValReservoir = scaler.transform(xValReservoir)
    xTestReservoir = scaler.transform(xTestReservoir)

    classifier = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=randomState,
    )
    classifier.fit(xTrainReservoir, yTrain)
    valProba = classifier.predict_proba(xValReservoir)[:, 1].astype(float)
    bestThreshold, bestValF1 = tuneThreshold(yVal, valProba)
    testProba = classifier.predict_proba(xTestReservoir)[:, 1].astype(float)

    modelMetrics, predictionDf, countDf = evaluateBinaryPredictions(
        modelName=modelName,
        family="Transfer Learning / Advanced",
        patientIds=testPatientIds,
        yTrue=yTest,
        yProb=testProba,
        threshold=bestThreshold,
    )
    modelMetrics["Validation F1"] = float(bestValF1)
    modelMetrics["Epochs Trained"] = 1
    return modelMetrics, predictionDf, countDf, {"reservoir_dim": 160, "state_features": int(xTrainReservoir.shape[1])}


### Step 21 - Train Transfer Learning and Advanced Models

Train `Transfer Learning`, `VICReg`, and `ESN`, then save the prediction output for each model.


In [ ]:
transferMetrics, transferPredDf, transferCountDf, transferHistory = trainTransferLearningModel(
    modelName="Transfer Learning (In-domain DAE to Classifier)",
    xTrain=xTrainSeq,
    yTrain=yTrainSeq,
    xVal=xValSeq,
    yVal=yValSeq,
    xTest=xTestSeq,
    yTest=yTestSeq,
    testPatientIds=testIds,
    pretrainEpochs=40,
    finetuneEpochs=60,
    batchSize=16,
)
benchmarkResults.append(transferMetrics)
benchmarkCountReports.append(transferCountDf)
benchmarkPredictionTables[transferMetrics["Model"]] = transferPredDf
benchmarkHistories[transferMetrics["Model"]] = transferHistory
print(
    f"{transferMetrics['Model']} | AUC={transferMetrics['AUC']:.4f} | F1={transferMetrics['F1']:.4f} | "
    f"Correct={transferMetrics['Correctly Predicted']}/{transferMetrics['Total Samples']}"
)
print("Prediction CSV:", transferMetrics["Prediction CSV"])

vicregMetrics, vicregPredDf, vicregCountDf, vicregHistory = trainVicRegModel(
    modelName="VICReg (Self-supervised)",
    xTrain=xTrainSeq,
    yTrain=yTrainSeq,
    xVal=xValSeq,
    yVal=yValSeq,
    xTest=xTestSeq,
    yTest=yTestSeq,
    testPatientIds=testIds,
    pretrainEpochs=30,
    finetuneEpochs=50,
    batchSize=16,
)
benchmarkResults.append(vicregMetrics)
benchmarkCountReports.append(vicregCountDf)
benchmarkPredictionTables[vicregMetrics["Model"]] = vicregPredDf
benchmarkHistories[vicregMetrics["Model"]] = vicregHistory
print(
    f"{vicregMetrics['Model']} | AUC={vicregMetrics['AUC']:.4f} | F1={vicregMetrics['F1']:.4f} | "
    f"Correct={vicregMetrics['Correctly Predicted']}/{vicregMetrics['Total Samples']}"
)
print("Prediction CSV:", vicregMetrics["Prediction CSV"])

esnMetrics, esnPredDf, esnCountDf, esnHistory = trainEsnModel(
    modelName="Echo State Network (ESN)",
    xTrain=xTrainSeq,
    yTrain=yTrainSeq,
    xVal=xValSeq,
    yVal=yValSeq,
    xTest=xTestSeq,
    yTest=yTestSeq,
    testPatientIds=testIds,
)
benchmarkResults.append(esnMetrics)
benchmarkCountReports.append(esnCountDf)
benchmarkPredictionTables[esnMetrics["Model"]] = esnPredDf
benchmarkHistories[esnMetrics["Model"]] = esnHistory
print(
    f"{esnMetrics['Model']} | AUC={esnMetrics['AUC']:.4f} | F1={esnMetrics['F1']:.4f} | "
    f"Correct={esnMetrics['Correctly Predicted']}/{esnMetrics['Total Samples']}"
)
print("Prediction CSV:", esnMetrics["Prediction CSV"])

print("\nTotal evaluated models:", len(benchmarkResults))


## 6. Validation

This section ranks the models, compares the metrics, and creates the evaluation visualizations. Because the dataset is imbalanced, the validation emphasizes PR-AUC, F1, sensitivity, balanced accuracy, and correct Brugada detection instead of plain accuracy alone.


### Step 22 - Build Leaderboard and Save Summary Files

Combine all model results, rank the leaderboard, and save `summary.csv`. The ranking is imbalance-aware and prioritizes PR-AUC, F1, sensitivity, correct Brugada detection, and balanced accuracy ahead of raw accuracy.


In [ ]:
benchmarkResultsDf = pd.DataFrame(benchmarkResults)
benchmarkResultsDf = benchmarkResultsDf.sort_values(
    by=["PR AUC", "F1", "Sensitivity", "Correct Brugada", "Balanced Accuracy", "AUC", "Specificity", "Accuracy", "Correctly Predicted"],
    ascending=[False, False, False, False, False, False, False, False, False],
).reset_index(drop=True)
benchmarkResultsDf.insert(0, "Rank", np.arange(1, len(benchmarkResultsDf) + 1))

benchmarkCountDf = pd.concat(benchmarkCountReports, axis=0).reset_index(drop=True)

summaryFile = projectRoot / "summary.csv"
benchmarkResultsDf.to_csv(summaryFile, index=False)

countSummaryFile = predictionDir / "model_count_comparison.csv"
benchmarkCountDf.to_csv(countSummaryFile, index=False)

print("Saved summary:", summaryFile)
print("Saved class count summary:", countSummaryFile)
print("Ranking priority: PR AUC -> F1 -> Sensitivity -> Correct Brugada -> Balanced Accuracy -> AUC -> Specificity -> Accuracy")
display(benchmarkResultsDf)
display(benchmarkCountDf)


### Step 23 - Create Validation Comparison Plots

Build the comparison plots for performance metrics, ROC curves, PR curves, and per-class counts. The visual focus remains on metrics that are informative under class imbalance.


In [ ]:
plt.figure(figsize=(14, 5))
sns.barplot(data=benchmarkResultsDf, x="Model", y="Correctly Predicted", hue="Family")
plt.title("Correct Predictions by Model")
plt.xlabel("")
plt.ylabel("Correctly predicted")
plt.xticks(rotation=30, ha="right")
save_plot("validation_correct_predictions_by_model")

metricHeatmapDf = benchmarkResultsDf.set_index("Model")[["AUC", "PR AUC", "F1", "Accuracy", "Sensitivity", "Specificity"]]
plt.figure(figsize=(10, 6))
sns.heatmap(metricHeatmapDf, annot=True, cmap="YlGnBu", vmin=0.0, vmax=1.0)
plt.title("Metric Heatmap Across Models")
save_plot("validation_metric_heatmap")

plt.figure(figsize=(10, 7))
for modelName, predDf in benchmarkPredictionTables.items():
    fpr, tpr, _ = roc_curve(predDf["target"], predDf["pred_proba"])
    aucValue = roc_auc_score(predDf["target"], predDf["pred_proba"])
    plt.plot(fpr, tpr, label=f"{modelName} (AUC={aucValue:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("ROC Curve Comparison on the Shared Test Split")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_plot("validation_roc_curve_comparison")

plt.figure(figsize=(10, 7))
for modelName, predDf in benchmarkPredictionTables.items():
    precisionCurve, recallCurve, _ = precision_recall_curve(predDf["target"], predDf["pred_proba"])
    prAucValue = average_precision_score(predDf["target"], predDf["pred_proba"])
    plt.plot(recallCurve, precisionCurve, label=f"{modelName} (PR-AUC={prAucValue:.3f})")
plt.title("Precision Recall Curve Comparison on the Shared Test Split")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_plot("validation_pr_curve_comparison")

countPlotDf = benchmarkCountDf[benchmarkCountDf["Class"] != "TOTAL"].copy()
plt.figure(figsize=(14, 5))
sns.barplot(data=countPlotDf, x="Model", y="Correctly Predicted", hue="Class")
plt.title("Correct Predictions per Class and Model")
plt.xlabel("")
plt.ylabel("Correctly predicted")
plt.xticks(rotation=30, ha="right")
save_plot("validation_correct_predictions_per_class")


### Step 24 - Inspect the Best Model

Display the confusion matrix for the best model and summarize its training history when available.


In [ ]:
bestRow = benchmarkResultsDf.iloc[0]
bestModelName = bestRow["Model"]
bestPredictionDf = benchmarkPredictionTables[bestModelName]
bestCountDf = benchmarkCountDf[benchmarkCountDf["Model"] == bestModelName].copy()

bestConfusion = confusion_matrix(bestPredictionDf["target"], bestPredictionDf["pred_label"], labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(bestConfusion, annot=True, fmt="d", cmap="Blues")
plt.title(f"Best Model Confusion Matrix: {bestModelName}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
save_plot("validation_best_model_confusion_matrix")

deepHistoryModels = [
    modelName
    for modelName, historyObj in benchmarkHistories.items()
    if isinstance(historyObj, dict) and "loss" in historyObj
]
if deepHistoryModels:
    plt.figure(figsize=(12, 5))
    for modelName in deepHistoryModels:
        historyObj = benchmarkHistories[modelName]
        val_auc_values = historyObj.get("val_auc", [])
        if val_auc_values:
            plt.plot(val_auc_values, label=modelName)
    plt.title("Validation AUC During Training")
    plt.xlabel("Epoch")
    plt.ylabel("Validation AUC")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    save_plot("validation_training_auc_history")

display(bestPredictionDf.head(20))
display(bestCountDf)


## 7. Summary

This final section prints the best-model decision and explains the main reason behind it. The final decision is imbalance-aware and gives special attention to Brugada detection quality.


### Step 25 - Print Final Decision

Display the best model, explain why it was selected, and show the location of all output files. The explanation explicitly references the imbalanced nature of the dataset.


In [ ]:
bestRow = benchmarkResultsDf.iloc[0]
secondRow = benchmarkResultsDf.iloc[1] if len(benchmarkResultsDf) > 1 else None

decisionReasons = [
    f"{bestRow['Model']} is ranked first because it leads the imbalance-aware ranking with PR-AUC={bestRow['PR AUC']:.4f}, "
    f"F1={bestRow['F1']:.4f}, Sensitivity={bestRow['Sensitivity']:.4f}, and {int(bestRow['Correct Brugada'])} correctly detected Brugada case(s)."
]

if secondRow is not None:
    decisionReasons.append(
        f"Compared with the runner-up, it improves Brugada detection by {int(bestRow['Correct Brugada'] - secondRow['Correct Brugada'])} case(s) and keeps a stronger balance on minority-class metrics."
    )
    if float(bestRow["PR AUC"]) >= float(secondRow["PR AUC"]):
        decisionReasons.append(
            f"Its PR-AUC of {bestRow['PR AUC']:.4f} is especially important here because Brugada is the minority class."
        )

if bestRow["Family"] == "Feature-Based":
    decisionReasons.append(
        "This suggests the handcrafted ECG morphology features generalize more reliably than the sequence models on this patient split."
    )
elif bestRow["Family"] == "Deep Learning End-to-End":
    decisionReasons.append(
        "This suggests the median beat representation carries enough spatial-temporal information for direct end-to-end learning to win on this split."
    )
else:
    decisionReasons.append(
        "This suggests the in-domain representation learning step helped the model make the most of the limited Brugada HUCA training data."
    )

print("Shared split fairness note:")
print("All 11 models below use the same patient split data so the comparison stays fair.")
print()
print("Imbalance note:")
print("The dataset is imbalanced, so the final ranking emphasizes PR-AUC, F1, sensitivity, balanced accuracy, and correct Brugada detection instead of plain accuracy alone.")
print()
print("Best model decision:")
print("Selected model:", bestRow["Model"])
print("Model family:", bestRow["Family"])
print("Reason:")
print(" ".join(decisionReasons))
print()
print("Best model metrics:")
print(
    f"AUC={bestRow['AUC']:.4f} | PR-AUC={bestRow['PR AUC']:.4f} | F1={bestRow['F1']:.4f} | "
    f"Balanced Accuracy={bestRow['Balanced Accuracy']:.4f} | Sensitivity={bestRow['Sensitivity']:.4f} | "
    f"Specificity={bestRow['Specificity']:.4f} | Correct Brugada={int(bestRow['Correct Brugada'])}"
)
print()
print("Saved outputs:")
print("summary.csv:", summaryFile)
print("prediction folder:", predictionDir)
print("plots folder:", plotDir)

display(benchmarkResultsDf)
